##Load the Bronze master data

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Load master datasets
customers = spark.table("workspace.bronze.customers")
accounts = spark.table("workspace.bronze.accounts")
merchants = spark.table("workspace.bronze.merchants")
devices = spark.table("workspace.bronze.devices")

print("Customers :", f"{customers.count():,}")
print("Accounts  :", f"{accounts.count():,}")
print("Merchants :", f"{merchants.count():,}")
print("Devices   :", f"{devices.count():,}")

Customers : 100,000
Accounts  : 150,000
Merchants : 25,000
Devices   : 75,000


In [0]:
account_pool = (
    accounts
    .select(
        "account_id",
        "customer_id"
    )
    .filter(F.col("status") == "ACTIVE")
)

print("Active accounts:", f"{account_pool.count():,}")

display(account_pool.limit(10))

Active accounts: 135,058


account_id,customer_id
ACC000001,CUST014003
ACC000003,CUST024143
ACC000004,CUST026129
ACC000006,CUST040901
ACC000007,CUST055285
ACC000008,CUST038473
ACC000009,CUST017541
ACC000010,CUST008838
ACC000011,CUST085956
ACC000012,CUST085681


In [0]:
merchant_pool = (
    merchants
    .select("merchant_id")
    .filter(F.col("status") == "ACTIVE")
)

device_pool = (
    devices
    .select(
        "device_id",
        "customer_id"
    )
    .filter(F.col("status") == "ACTIVE")
)

print("Active merchants:", f"{merchant_pool.count():,}")
print("Active devices  :", f"{device_pool.count():,}")

Active merchants: 23,986
Active devices  : 69,712


##transaction configuration

In [0]:
TARGET_TRANSACTIONS = 5_000_000

TRANSACTION_TYPES = [
    "DEBIT",
    "CREDIT"
]

CHANNELS = [
    "UPI",
    "ATM",
    "CARD",
    "NETBANKING",
    "MOBILE",
    "BRANCH"
]

CURRENCIES = [
    "INR"
]

STATUSES = [
    "SUCCESS",
    "FAILED",
    "PENDING"
]

print("Target transactions:", f"{TARGET_TRANSACTIONS:,}")

Target transactions: 5,000,000


##Create Indexed Master Pools


In [0]:
from pyspark.sql.window import Window

account_window = Window.orderBy("account_id")

account_pool_indexed = (
    account_pool
    .withColumn(
        "account_index",
        F.row_number().over(account_window) - 1
    )
)

ACCOUNT_COUNT = account_pool_indexed.count()

print("Indexed active accounts:", f"{ACCOUNT_COUNT:,}")

display(account_pool_indexed.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Indexed active accounts: 135,058


account_id,customer_id,account_index
ACC000001,CUST014003,0
ACC000003,CUST024143,1
ACC000004,CUST026129,2
ACC000006,CUST040901,3
ACC000007,CUST055285,4
ACC000008,CUST038473,5
ACC000009,CUST017541,6
ACC000010,CUST008838,7
ACC000011,CUST085956,8
ACC000012,CUST085681,9


In [0]:
merchant_window = Window.orderBy("merchant_id")

merchant_pool_indexed = (
    merchant_pool
    .withColumn(
        "merchant_index",
        F.row_number().over(merchant_window) - 1
    )
)

MERCHANT_COUNT = merchant_pool_indexed.count()

print("Indexed active merchants:", f"{MERCHANT_COUNT:,}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Indexed active merchants: 23,986


In [0]:
device_window = Window.orderBy("device_id")

device_pool_indexed = (
    device_pool
    .withColumn(
        "device_index",
        F.row_number().over(device_window) - 1
    )
)

DEVICE_COUNT = device_pool_indexed.count()

print("Indexed active devices:", f"{DEVICE_COUNT:,}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Indexed active devices: 69,712


##Generate the test transactions

In [0]:
TEST_TRANSACTIONS = 100_000

transactions_test = (
    spark.range(TEST_TRANSACTIONS)
    .withColumn(
        "txn_id",
        F.concat(
            F.lit("TXN"),
            F.lpad(
                (F.col("id") + 1).cast("string"),
                10,
                "0"
            )
        )
    )
    .withColumn(
        "account_index",
        F.pmod(F.col("id"), F.lit(ACCOUNT_COUNT))
    )
    .withColumn(
        "merchant_index",
        F.pmod(F.col("id") * 7, F.lit(MERCHANT_COUNT))
    )
    .withColumn(
        "device_index",
        F.pmod(F.col("id") * 13, F.lit(DEVICE_COUNT))
    )
)

##Attach Information

In [0]:
transactions_test = (
    transactions_test
    .join(
        account_pool_indexed.select(
            "account_index",
            "account_id",
            "customer_id"
        ),
        "account_index",
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_test = (
    transactions_test
    .join(
        merchant_pool_indexed.select(
            "merchant_index",
            "merchant_id"
        ),
        "merchant_index",
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_test = (
    transactions_test
    .join(
        device_pool_indexed.select(
            "device_index",
            "device_id"
        ),
        "device_index",
        "left"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##Generate Transaction Attributes

In [0]:
transactions_test = (
    transactions_test
    .withColumn(
        "txn_time",
        F.expr("""
            timestampadd(
                SECOND,
                CAST(rand(42) * 31536000 AS INT),
                TIMESTAMP('2025-01-01 00:00:00')
            )
        """)
    )
    .withColumn(
        "txn_type",
        F.when(F.rand(10) < 0.55, "DEBIT")
         .otherwise("CREDIT")
    )
    .withColumn(
        "channel",
        F.when(F.rand(20) < 0.35, "UPI")
         .when(F.rand(21) < 0.55, "CARD")
         .when(F.rand(22) < 0.70, "NETBANKING")
         .when(F.rand(23) < 0.82, "MOBILE")
         .when(F.rand(24) < 0.94, "ATM")
         .otherwise("BRANCH")
    )
    .withColumn(
        "currency",
        F.lit("INR")
    )
    .withColumn(
        "status",
        F.when(F.rand(30) < 0.94, "SUCCESS")
         .when(F.rand(31) < 0.98, "FAILED")
         .otherwise("PENDING")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_test = (
    transactions_test
    .withColumn(
        "amount",
        F.when(
            F.col("channel") == "UPI",
            F.round(F.rand(40) * 49_990 + 10, 2)
        )
        .when(
            F.col("channel") == "ATM",
            F.round(F.rand(41) * 49_500 + 500, 2)
        )
        .when(
            F.col("channel") == "CARD",
            F.round(F.rand(42) * 199_900 + 100, 2)
        )
        .when(
            F.col("channel") == "NETBANKING",
            F.round(F.rand(43) * 499_900 + 100, 2)
        )
        .when(
            F.col("channel") == "MOBILE",
            F.round(F.rand(44) * 99_990 + 10, 2)
        )
        .otherwise(
            F.round(F.rand(45) * 999_500 + 500, 2)
        )
        .cast("decimal(18,2)")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


##Final Schema

In [0]:
transactions_test = (
    transactions_test
    .select(
        "txn_id",
        "account_id",
        "customer_id",
        "merchant_id",
        "device_id",
        "txn_time",
        "txn_type",
        "amount",
        "currency",
        "channel",
        "status"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(transactions_test.orderBy("txn_id").limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status
TXN0000000001,ACC000001,CUST014003,MER00001,DEV000001,2025-02-01T07:13:08.000Z,CREDIT,279416.67,INR,NETBANKING,FAILED
TXN0000000002,ACC000003,CUST024143,MER00009,DEV000015,2025-04-24T07:12:13.000Z,DEBIT,326565.73,INR,NETBANKING,SUCCESS
TXN0000000003,ACC000004,CUST026129,MER00016,DEV000031,2025-01-23T20:06:40.000Z,DEBIT,17242.54,INR,CARD,SUCCESS
TXN0000000004,ACC000006,CUST040901,MER00023,DEV000044,2025-04-22T20:36:04.000Z,DEBIT,62151.24,INR,CARD,SUCCESS
TXN0000000005,ACC000007,CUST055285,MER00030,DEV000057,2025-01-01T00:22:34.000Z,CREDIT,34589.57,INR,ATM,SUCCESS
TXN0000000006,ACC000008,CUST038473,MER00037,DEV000071,2025-04-21T14:51:07.000Z,CREDIT,12607.70,INR,CARD,SUCCESS
TXN0000000007,ACC000009,CUST017541,MER00044,DEV000084,2025-02-01T23:27:36.000Z,DEBIT,12453.61,INR,UPI,SUCCESS
TXN0000000008,ACC000010,CUST008838,MER00052,DEV000099,2025-03-24T20:48:55.000Z,CREDIT,44522.85,INR,UPI,SUCCESS
TXN0000000009,ACC000011,CUST085956,MER00059,DEV000114,2025-11-11T22:05:17.000Z,DEBIT,348436.74,INR,NETBANKING,SUCCESS
TXN0000000010,ACC000012,CUST085681,MER00067,DEV000130,2025-07-10T23:42:16.000Z,CREDIT,23135.04,INR,UPI,SUCCESS


In [0]:
print("Total transactions:", f"{transactions_test.count():,}")

print(
    "Unique transaction IDs:",
    f"{transactions_test.select('txn_id').distinct().count():,}"
)

print(
    "Null account IDs:",
    transactions_test.filter(F.col("account_id").isNull()).count()
)

print(
    "Null customer IDs:",
    transactions_test.filter(F.col("customer_id").isNull()).count()
)

print(
    "Null merchant IDs:",
    transactions_test.filter(F.col("merchant_id").isNull()).count()
)

print(
    "Null device IDs:",
    transactions_test.filter(F.col("device_id").isNull()).count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total transactions: 100,000
Unique transaction IDs: 100,000
Null account IDs: 0
Null customer IDs: 0


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Null merchant IDs: 0
Null device IDs: 0


In [0]:
txn_check = transactions_test.alias("t")

account_check = accounts.select(
    "account_id",
    "customer_id"
).alias("a")

invalid_customer_mapping = (
    txn_check
    .join(
        account_check,
        F.col("t.account_id") == F.col("a.account_id"),
        "left"
    )
    .filter(
        F.col("t.customer_id") != F.col("a.customer_id")
    )
)

invalid_count = invalid_customer_mapping.count()

print("Invalid account/customer relationships:", invalid_count)

assert invalid_count == 0

print("PASSED")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Invalid account/customer relationships: 0
PASSED


##Basic Transaction Quality

In [0]:
quality = transactions_test.select(
    F.count("*").alias("total"),
    F.countDistinct("txn_id").alias("unique_txn_ids"),
    F.sum(
        F.when(F.col("amount") <= 0, 1).otherwise(0)
    ).alias("invalid_amounts"),
    F.sum(
        F.when(~F.col("txn_type").isin("DEBIT", "CREDIT"), 1)
         .otherwise(0)
    ).alias("invalid_types"),
    F.sum(
        F.when(F.col("currency") != "INR", 1)
         .otherwise(0)
    ).alias("invalid_currency")
)

display(quality)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


total,unique_txn_ids,invalid_amounts,invalid_types,invalid_currency
100000,100000,0,0,0


##Scale to 5 Million

In [0]:
TARGET_TRANSACTIONS = 5_000_000

transactions_base = (
    spark.range(TARGET_TRANSACTIONS)
    .withColumn(
        "txn_id",
        F.concat(
            F.lit("TXN"),
            F.lpad(
                (F.col("id") + 1).cast("string"),
                10,
                "0"
            )
        )
    )
    .withColumn(
        "account_index",
        F.pmod(F.col("id"), F.lit(ACCOUNT_COUNT))
    )
    .withColumn(
        "merchant_index",
        F.pmod(F.col("id") * 7, F.lit(MERCHANT_COUNT))
    )
    .withColumn(
        "device_index",
        F.pmod(F.col("id") * 13, F.lit(DEVICE_COUNT))
    )
)

print("Generated base rows:", f"{transactions_base.count():,}")

Generated base rows: 5,000,000


In [0]:
transactions_base = (
    transactions_base
    .join(
        account_pool_indexed.select(
            "account_index",
            "account_id",
            "customer_id"
        ),
        "account_index",
        "left"
    )
)

print("Account/customer mapping completed")

Account/customer mapping completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_base = (
    transactions_base
    .join(
        merchant_pool_indexed.select(
            "merchant_index",
            "merchant_id"
        ),
        "merchant_index",
        "left"
    )
)

print("Merchant mapping completed")

Merchant mapping completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_base = (
    transactions_base
    .join(
        device_pool_indexed.select(
            "device_index",
            "device_id"
        ),
        "device_index",
        "left"
    )
)

print("Device mapping completed")

Device mapping completed


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_base = (
    transactions_base
    .withColumn(
        "txn_time",
        F.expr("""
            timestampadd(
                SECOND,
                CAST(rand(42) * 31536000 AS INT),
                TIMESTAMP('2025-01-01 00:00:00')
            )
        """)
    )
    .withColumn(
        "txn_type",
        F.when(F.rand(10) < 0.55, "DEBIT")
         .otherwise("CREDIT")
    )
    .withColumn(
        "channel",
        F.when(F.rand(20) < 0.35, "UPI")
         .when(F.rand(21) < 0.55, "CARD")
         .when(F.rand(22) < 0.70, "NETBANKING")
         .when(F.rand(23) < 0.82, "MOBILE")
         .when(F.rand(24) < 0.94, "ATM")
         .otherwise("BRANCH")
    )
    .withColumn(
        "currency",
        F.lit("INR")
    )
    .withColumn(
        "status",
        F.when(F.rand(30) < 0.94, "SUCCESS")
         .when(F.rand(31) < 0.98, "FAILED")
         .otherwise("PENDING")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_base = (
    transactions_base
    .withColumn(
        "amount",
        F.when(
            F.col("channel") == "UPI",
            F.round(F.rand(40) * 49_990 + 10, 2)
        )
        .when(
            F.col("channel") == "ATM",
            F.round(F.rand(41) * 49_500 + 500, 2)
        )
        .when(
            F.col("channel") == "CARD",
            F.round(F.rand(42) * 199_900 + 100, 2)
        )
        .when(
            F.col("channel") == "NETBANKING",
            F.round(F.rand(43) * 499_900 + 100, 2)
        )
        .when(
            F.col("channel") == "MOBILE",
            F.round(F.rand(44) * 99_990 + 10, 2)
        )
        .otherwise(
            F.round(F.rand(45) * 999_500 + 500, 2)
        )
        .cast("decimal(18,2)")
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_base = (
    transactions_base
    .select(
        "txn_id",
        "account_id",
        "customer_id",
        "merchant_id",
        "device_id",
        "txn_time",
        "txn_type",
        "amount",
        "currency",
        "channel",
        "status"
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
print("Total transactions:", f"{transactions_base.count():,}")

display(
    transactions_base.limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total transactions: 5,000,000


txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status
TXN0000000001,ACC000001,CUST014003,MER00001,DEV000001,2025-02-01T07:13:08.000Z,CREDIT,279416.67,INR,NETBANKING,FAILED
TXN0000000002,ACC000003,CUST024143,MER00009,DEV000015,2025-04-24T07:12:13.000Z,DEBIT,326565.73,INR,NETBANKING,SUCCESS
TXN0000000003,ACC000004,CUST026129,MER00016,DEV000031,2025-01-23T20:06:40.000Z,DEBIT,17242.54,INR,CARD,SUCCESS
TXN0000000004,ACC000006,CUST040901,MER00023,DEV000044,2025-04-22T20:36:04.000Z,DEBIT,62151.24,INR,CARD,SUCCESS
TXN0000000005,ACC000007,CUST055285,MER00030,DEV000057,2025-01-01T00:22:34.000Z,CREDIT,34589.57,INR,ATM,SUCCESS
TXN0000000006,ACC000008,CUST038473,MER00037,DEV000071,2025-04-21T14:51:07.000Z,CREDIT,12607.70,INR,CARD,SUCCESS
TXN0000000007,ACC000009,CUST017541,MER00044,DEV000084,2025-02-01T23:27:36.000Z,DEBIT,12453.61,INR,UPI,SUCCESS
TXN0000000008,ACC000010,CUST008838,MER00052,DEV000099,2025-03-24T20:48:55.000Z,CREDIT,44522.85,INR,UPI,SUCCESS
TXN0000000009,ACC000011,CUST085956,MER00059,DEV000114,2025-11-11T22:05:17.000Z,DEBIT,348436.74,INR,NETBANKING,SUCCESS
TXN0000000010,ACC000012,CUST085681,MER00067,DEV000130,2025-07-10T23:42:16.000Z,CREDIT,23135.04,INR,UPI,SUCCESS


##final source transactions

In [0]:
transactions_base = transactions_base.repartition(50)

print(
    "Clean transaction count:",
    f"{transactions_base.count():,}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Clean transaction count: 5,000,000


##final transaction-quality transformation.

In [0]:
transactions_final = (
    transactions_base
    .withColumn(
        "_quality_bucket",
        F.when(
            F.pmod(F.hash("txn_id"), F.lit(1000)) == 0,
            "INVALID_AMOUNT"
        )
        .when(
            F.pmod(F.hash("txn_id"), F.lit(1000)) == 1,
            "INVALID_CURRENCY"
        )
        .when(
            F.pmod(F.hash("txn_id"), F.lit(1000)) == 2,
            "INVALID_ACCOUNT"
        )
        .when(
            F.pmod(F.hash("txn_id"), F.lit(1000)) == 3,
            "INVALID_CUSTOMER"
        )
        .when(
            F.pmod(F.hash("txn_id"), F.lit(250)) == 5,
            "HIGH_VALUE"
        )
        .when(
            F.pmod(F.hash("txn_id"), F.lit(300)) == 7,
            "LATE_EVENT"
        )
        .otherwise("NORMAL")
    )
)

display(
    transactions_final
    .groupBy("_quality_bucket")
    .count()
    .orderBy(F.desc("count"))
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


_quality_bucket,count
NORMAL,4943573
HIGH_VALUE,19873
LATE_EVENT,16644
INVALID_AMOUNT,4992
INVALID_CURRENCY,4983
INVALID_CUSTOMER,4983
INVALID_ACCOUNT,4952


In [0]:
transactions_final = (
    transactions_final
    .withColumn(
        "amount",
        F.when(
            F.col("_quality_bucket") == "INVALID_AMOUNT",
            F.when(
                F.pmod(F.hash("txn_id"), F.lit(2)) == 0,
                F.lit(0).cast("decimal(18,2)")
            ).otherwise(
                F.lit(-100).cast("decimal(18,2)")
            )
        )
        .when(
            F.col("_quality_bucket") == "HIGH_VALUE",
            F.round(
                F.rand(200) * 900_000 + 100_000,
                2
            ).cast("decimal(18,2)")
        )
        .otherwise(F.col("amount"))
    )
    .withColumn(
        "currency",
        F.when(
            F.col("_quality_bucket") == "INVALID_CURRENCY",
            F.lit("XXX")
        ).otherwise(F.col("currency"))
    )
    .withColumn(
        "account_id",
        F.when(
            F.col("_quality_bucket") == "INVALID_ACCOUNT",
            F.concat(
                F.lit("INVALID_ACC_"),
                F.substring("txn_id", 4, 10)
            )
        ).otherwise(F.col("account_id"))
    )
    .withColumn(
        "customer_id",
        F.when(
            F.col("_quality_bucket") == "INVALID_CUSTOMER",
            F.concat(
                F.lit("INVALID_CUST_"),
                F.substring("txn_id", 4, 10)
            )
        ).otherwise(F.col("customer_id"))
    )
    .withColumn(
        "txn_time",
        F.when(
            F.col("_quality_bucket") == "LATE_EVENT",
            F.expr("timestampadd(DAY, -30, txn_time)")
        ).otherwise(F.col("txn_time"))
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
transactions_final = transactions_final.drop("_quality_bucket")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
final_count = transactions_final.count()

print("Final transaction count:", f"{final_count:,}")

assert final_count == 5_000_000

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Final transaction count: 5,000,000


##Inspect the quality issues

In [0]:
print("Invalid amounts:",
      transactions_final.filter(F.col("amount") <= 0).count())

print("Invalid currencies:",
      transactions_final.filter(F.col("currency") != "INR").count())

print("Invalid accounts:",
      transactions_final.filter(
          F.col("account_id").startswith("INVALID_ACC_")
      ).count())

print("Invalid customers:",
      transactions_final.filter(
          F.col("customer_id").startswith("INVALID_CUST_")
      ).count())

print("High-value transactions:",
      transactions_final.filter(
          F.col("amount") >= 100_000
      ).count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Invalid amounts: 4992


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Invalid currencies: 4983
Invalid accounts: 4952
Invalid customers: 4983
High-value transactions: 1727716


In [0]:
display(
    transactions_final
    .orderBy("txn_id")
    .limit(20)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


txn_id,account_id,customer_id,merchant_id,device_id,txn_time,txn_type,amount,currency,channel,status
TXN0000000001,ACC000001,CUST014003,MER00001,DEV000001,2025-02-01T07:13:08.000Z,CREDIT,279416.67,INR,NETBANKING,FAILED
TXN0000000002,ACC000003,CUST024143,MER00009,DEV000015,2025-04-24T07:12:13.000Z,DEBIT,326565.73,INR,NETBANKING,SUCCESS
TXN0000000003,ACC000004,CUST026129,MER00016,DEV000031,2025-01-23T20:06:40.000Z,DEBIT,17242.54,INR,CARD,SUCCESS
TXN0000000004,ACC000006,CUST040901,MER00023,DEV000044,2025-04-22T20:36:04.000Z,DEBIT,62151.24,INR,CARD,SUCCESS
TXN0000000005,ACC000007,CUST055285,MER00030,DEV000057,2025-01-01T00:22:34.000Z,CREDIT,34589.57,INR,ATM,SUCCESS
TXN0000000006,ACC000008,CUST038473,MER00037,DEV000071,2025-04-21T14:51:07.000Z,CREDIT,12607.70,INR,CARD,SUCCESS
TXN0000000007,ACC000009,CUST017541,MER00044,DEV000084,2025-02-01T23:27:36.000Z,DEBIT,12453.61,INR,UPI,SUCCESS
TXN0000000008,ACC000010,CUST008838,MER00052,DEV000099,2025-03-24T20:48:55.000Z,CREDIT,44522.85,INR,UPI,SUCCESS
TXN0000000009,ACC000011,CUST085956,MER00059,DEV000114,2025-11-11T22:05:17.000Z,DEBIT,348436.74,INR,NETBANKING,SUCCESS
TXN0000000010,ACC000012,CUST085681,MER00067,DEV000130,2025-07-10T23:42:16.000Z,CREDIT,23135.04,INR,UPI,SUCCESS


In [0]:
transactions_final = (
    transactions_final
    .withColumn(
        "_fraud_scenario",
        F.when(
            F.pmod(F.hash("txn_id"), F.lit(250)) == 5,
            F.lit("HIGH_VALUE")
        ).otherwise(
            F.lit("NORMAL")
        )
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
display(
    transactions_final
    .groupBy("_fraud_scenario")
    .count()
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


_fraud_scenario,count
HIGH_VALUE,19873
NORMAL,4980127


In [0]:
duplicate_transactions = (
    transactions_final
    .filter(
        F.pmod(F.hash("txn_id"), F.lit(500)) == 11
    )
    .limit(10_000)
)

print(
    "Duplicate records prepared:",
    f"{duplicate_transactions.count():,}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Duplicate records prepared: 10,000


In [0]:
transactions_source = (
    transactions_final
    .unionByName(duplicate_transactions)
)

print(
    "Source transaction rows:",
    f"{transactions_source.count():,}"
)

print(
    "Distinct transaction IDs:",
    f"{transactions_source.select('txn_id').distinct().count():,}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Source transaction rows: 5,010,000
Distinct transaction IDs: 5,000,000


In [0]:
transactions_source.printSchema()

root
 |-- txn_id: string (nullable = false)
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- txn_time: timestamp (nullable = true)
 |-- txn_type: string (nullable = false)
 |-- amount: decimal(18,2) (nullable = true)
 |-- currency: string (nullable = false)
 |-- channel: string (nullable = false)
 |-- status: string (nullable = false)
 |-- _fraud_scenario: string (nullable = false)



In [0]:
transactions_source = transactions_source.drop("_fraud_scenario")

print("Final source schema:")
transactions_source.printSchema()

Final source schema:


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


root
 |-- txn_id: string (nullable = false)
 |-- account_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- txn_time: timestamp (nullable = true)
 |-- txn_type: string (nullable = false)
 |-- amount: decimal(18,2) (nullable = true)
 |-- currency: string (nullable = false)
 |-- channel: string (nullable = false)
 |-- status: string (nullable = false)



In [0]:
print("Total source rows:",
      f"{transactions_source.count():,}")

print("Unique transaction IDs:",
      f"{transactions_source.select('txn_id').distinct().count():,}")

print(
    "Duplicate rows:",
    f"{transactions_source.count() - transactions_source.select('txn_id').distinct().count():,}"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total source rows: 5,010,000
Unique transaction IDs: 5,000,000
Duplicate rows: 10,000


In [0]:
EXPORT_PATH = "/Volumes/workspace/bronze/banking_source/transactions"

print(EXPORT_PATH)

/Volumes/workspace/bronze/banking_source/transactions


In [0]:
(
    transactions_source
    .repartition(51)
    .write
    .mode("overwrite")
    .option("header", "true")
    .option("quote", '"')
    .option("escape", '"')
    .csv(EXPORT_PATH)
)

print("Transaction CSV export completed")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Transaction CSV export completed


In [0]:
display(
    dbutils.fs.ls(EXPORT_PATH)
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/banking_source/transactions/_SUCCESS,_SUCCESS,0,1787381389000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/_committed_1365459471596642376,_committed_1365459471596642376,9214,1787381388000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/_committed_6862856986062073034,_committed_6862856986062073034,4614,1786944825000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/_committed_vacuum7726691950665307346,_committed_vacuum7726691950665307346,96,1787381389000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/_started_1365459471596642376,_started_1365459471596642376,0,1787381383000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00000-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2886-1-c000.csv,part-00000-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2886-1-c000.csv,11067918,1787381384000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00001-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2887-1-c000.csv,part-00001-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2887-1-c000.csv,11068255,1787381384000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00002-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2888-1-c000.csv,part-00002-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2888-1-c000.csv,11065238,1787381384000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00003-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2889-1-c000.csv,part-00003-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2889-1-c000.csv,11066417,1787381384000
dbfs:/Volumes/workspace/bronze/banking_source/transactions/part-00004-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2890-1-c000.csv,part-00004-tid-1365459471596642376-ed68690c-275e-4fcf-bb9d-a92032c0c1b5-2890-1-c000.csv,11067227,1787381384000
